# A2 · Step 1: generate synthetic gesture clips (text-to-video)

**SOTA idea:** Step-Video-T2V (Stepfun, 30B parameters, Video-VAE with 16×16 spatial and 8× temporal compression).
We use it to generate short hand-gesture clips with varied backgrounds, lighting and camera motion,
cut them into frames, and use the frames (a) to augment training and (b) as a stress-test set.

The full Step-Video-T2V needs an 80 GB GPU, so there are three ways to get clips:

| Option | What runs | Cost / hardware |
|---|---|---|
| **A. `wan`** | Open T2V model **Wan2.1-T2V-1.3B** in this notebook (lighter alternative; Step-Video idea credited) | Colab **L4/A100** or **Kaggle** (29 GB RAM). Free Colab T4 may run out of RAM. |
| **B. `replicate`** | Real **Step-Video-T2V** through the Replicate API | Paid per run, needs a Replicate token |
| **C. `manual`** | Clips made in any online Step-Video / T2V demo, uploaded to Drive | Free |

All options end the same way: `.mp4` files in `CLIPS_DIR/<class>/`. The main notebook starts from there.
⚠️ T2V models often draw hands wrongly. The main notebook has a manual check step: keep only clips where the gesture is correct.

In [ ]:
GENERATOR = "wan"          # "wan" | "replicate" | "manual"
CLIPS_PER_CLASS = 12       # 12 x 3 classes = 36 clips
RUN_ON = "colab"           # "colab" (saves to Drive) | "kaggle" (saves to /kaggle/working, download after)

import os
if RUN_ON == "colab":
    from google.colab import drive
    drive.mount('/content/drive')
    CLIPS_DIR = "/content/drive/MyDrive/comp-vis1/a2/synthetic_clips"
else:
    CLIPS_DIR = "/kaggle/working/synthetic_clips"
os.makedirs(CLIPS_DIR, exist_ok=True)
print("Clips will be saved to", CLIPS_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Clips will be saved to /content/drive/MyDrive/comp-vis1/a2/synthetic_clips


## Prompts
Each prompt = precise gesture description × environment × camera motion. The environment tag is saved so we can report accuracy per environment.

In [ ]:
import itertools, random, csv

GESTURES = {
    "peace-sign": "a human hand making a peace sign, index and middle fingers raised in a V shape, other fingers folded",
    "thumbs-up":  "a human hand giving a thumbs up, closed fist with the thumb pointing straight up",
    "stop":       "a human hand showing a stop gesture, open palm facing the camera, all five fingers extended and held together",
}
ENVIRONMENTS = {
    "dark_room":   "in a dark room lit only by a dim phone screen, low exposure",
    "sunny_street":"outdoors on a sunny city street, harsh sunlight",
    "kitchen":     "in a cluttered home kitchen, warm indoor light",
    "office":      "in an office with cold fluorescent light",
    "night_neon":  "outdoors at night under colorful neon signs",
    "plain_wall":  "in front of a plain white wall, soft light",
    "classroom":   "in a university classroom with desks in the background",
    "car":         "inside a car, daylight through the windows",
}
CAMERA = {
    "static":   "static camera",
    "pan":      "slow camera pan from left to right",
    "handheld": "handheld shaky camera, slight motion blur",
    "zoom":     "slow zoom in towards the hand",
}
STYLE = "realistic smartphone video, close-up, the hand is centered and fully visible, natural skin, 4 seconds"
NEGATIVE = ("extra fingers, missing fingers, deformed hand, fused fingers, blurry hand, cartoon, "
            "text, watermark, low quality, distorted anatomy")

random.seed(42)
jobs = []
combos = list(itertools.product(ENVIRONMENTS, CAMERA))
for cls, g in GESTURES.items():
    for i, (env, cam) in enumerate(random.sample(combos, CLIPS_PER_CLASS)):
        prompt = f"{g}, {ENVIRONMENTS[env]}, {CAMERA[cam]}, {STYLE}"
        jobs.append(dict(cls=cls, env=env, camera=cam, prompt=prompt,
                         file=f"{CLIPS_DIR}/{cls}/{cls}__{env}__{cam}__{i:02d}.mp4"))

with open(f"{CLIPS_DIR}/prompts.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=jobs[0].keys()); w.writeheader(); w.writerows(jobs)
print(len(jobs), "jobs. Example:\n", jobs[0]["prompt"])

36 jobs. Example:
 a human hand making a peace sign, index and middle fingers raised in a V shape, other fingers folded, outdoors on a sunny city street, harsh sunlight, slow zoom in towards the hand, realistic smartphone video, close-up, the hand is centered and fully visible, natural skin, 4 seconds


## Option A: Wan2.1-T2V-1.3B (open model, runs on free Colab T4)
Two stages, because free Colab has ~12.7 GB RAM and a 15 GB T4:
1. **Stage 1** encodes all 36 prompts with the text encoder (loaded straight to the GPU) and saves the embeddings to Drive.
2. **Restart the session** (Runtime → Restart session) to free GPU memory, run the first two cells again, skip `pip install`.
3. **Stage 2** loads only the video transformer + VAE in float16 and generates 384×672, 33-frame clips (~2 s at 16 fps), ~233 s per clip on a T4 (≈2.3 h for 36 clips).

Finished clips are skipped, so after a disconnect just re-run the cells.

In [ ]:
if GENERATOR == "wan":
    !pip install -q -U diffusers transformers accelerate ftfy imageio imageio-ffmpeg

In [ ]:
# Stage 1: encode all prompts with the umT5 text encoder, loaded straight to the GPU
# (free Colab has ~12.7 GB RAM; loading the full pipeline at once runs out of RAM).
# Embeddings are saved to Drive, so this runs only once.
import os, gc, torch
from diffusers import WanPipeline
from transformers import UMT5EncoderModel, AutoTokenizer

model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
EMB_DIR = f"{CLIPS_DIR}/_embeds"; os.makedirs(EMB_DIR, exist_ok=True)
need = [j for j in jobs if not os.path.exists(j["file"])
        and not os.path.exists(f"{EMB_DIR}/{os.path.basename(j['file'])}.pt")]
if need:
    te = UMT5EncoderModel.from_pretrained(model_id, subfolder="text_encoder",
                                          torch_dtype=torch.bfloat16, device_map="cuda")
    tok = AutoTokenizer.from_pretrained(model_id, subfolder="tokenizer")
    enc = WanPipeline.from_pretrained(model_id, text_encoder=te, tokenizer=tok,
                                      transformer=None, vae=None, torch_dtype=torch.bfloat16)
    with torch.no_grad():
        for j in need:
            pe, ne = enc.encode_prompt(prompt=j["prompt"], negative_prompt=NEGATIVE,
                                       device="cuda", dtype=torch.bfloat16)
            torch.save({"pe": pe.cpu(), "ne": ne.cpu()},
                       f"{EMB_DIR}/{os.path.basename(j['file'])}.pt")
print("Encoded prompts:", len(need), "| now: Runtime -> Restart session, run the first two cells, then Stage 2")


In [ ]:
# Stage 2: generate the clips (text encoder not loaded)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch, time, gc
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video

model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
EMB_DIR = f"{CLIPS_DIR}/_embeds"
todo = [j for j in jobs if not os.path.exists(j["file"])]
print("Осталось сгенерировать:", len(todo))

vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(model_id, vae=vae, text_encoder=None, tokenizer=None,
                                   torch_dtype=torch.float16).to("cuda")
pipe.vae.enable_tiling()
print("GPU:", torch.cuda.get_device_name(0))

for j in todo:
    e = torch.load(f"{EMB_DIR}/{os.path.basename(j['file'])}.pt")
    os.makedirs(os.path.dirname(j["file"]), exist_ok=True)
    t0 = time.time()
    with torch.no_grad():
        frames = pipe(prompt_embeds=e["pe"].to("cuda", torch.float16),
                      negative_prompt_embeds=e["ne"].to("cuda", torch.float16),
                      height=384, width=672, num_frames=33,
                      num_inference_steps=30, guidance_scale=5.0,
                      generator=torch.Generator("cuda").manual_seed(42)).frames[0]
    export_to_video(frames, j["file"], fps=16)
    print(f"{os.path.basename(j['file'])}  {time.time()-t0:.0f}s")
    gc.collect(); torch.cuda.empty_cache()

Осталось сгенерировать: 36


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/3 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

GPU: Tesla T4


  0%|          | 0/30 [00:00<?, ?it/s]

... (67 outputs left out)

  0%|          | 0/30 [00:00<?, ?it/s]

stop__office__pan__10.mp4  232s


  0%|          | 0/30 [00:00<?, ?it/s]

stop__kitchen__handheld__11.mp4  233s


## Option B: real Step-Video-T2V via Replicate
Open https://replicate.com/zsxkib/step-video-t2v → *API* tab, copy the full version id (`zsxkib/step-video-t2v:<hash>`) and check the input names there. Each run is billed by Replicate.

In [ ]:
if GENERATOR == "replicate":
    !pip install -q replicate
    import os, replicate, urllib.request
    os.environ["REPLICATE_API_TOKEN"] = "r8_..."                  # your token
    STEPVIDEO_VERSION = "zsxkib/step-video-t2v:PASTE_VERSION_HASH"  # from the API tab

    for j in jobs:
        if os.path.exists(j["file"]):
            continue
        os.makedirs(os.path.dirname(j["file"]), exist_ok=True)
        out = replicate.run(STEPVIDEO_VERSION, input={"prompt": j["prompt"],
                                                      "negative_prompt": NEGATIVE})
        url = out[0] if isinstance(out, list) else out
        url = getattr(url, "url", url)
        urllib.request.urlretrieve(str(url), j["file"])
        print("saved", j["file"])

## Option C: manual
Generate clips in any online T2V demo using the prompts from `prompts.csv`, and upload them to `CLIPS_DIR/<class>/` with the file names from the `file` column (the environment tag is read from the name).

In [ ]:
from pathlib import Path
for cls in GESTURES:
    n = len(list(Path(CLIPS_DIR, cls).glob("*.mp4")))
    print(f"{cls}: {n} clips")

peace-sign: 12 clips
thumbs-up: 12 clips
stop: 12 clips
